## 数据准备与特征工程 (Data Preparation & Feature Engineering)

**【中文说明】**
在此步骤中，我们将加载已对齐维度的训练集 (Chicago 2015-2024)、本地测试集 (Chicago 2025) 以及泛化测试集 (NIBRS)。
为了让逻辑回归模型更好地理解数据，我们执行了以下处理：
1. **时间循环编码 (Cyclical Encoding)**：将 `hour`, `month`, `weekday` 转换为正弦 (sin) 和余弦 (cos) 值，保留时间的周期性。
2. **独热编码 (One-Hot Encoding)**：将地点 (`location_name`)、犯罪类别 (`offense_category_name`) 等分类型变量转换为 0/1 矩阵。
3. **严格特征对齐 (Strict Feature Alignment)**：以训练集的特征列为基准，强制对齐两个测试集的特征列。如果测试集中缺失某类犯罪或地点，则自动填充 0，确保模型在跨数据集预测时矩阵维度绝对一致。

**【English Explanation】**
In this step, we load the aligned training set (Chicago 2015-2024), local test set (Chicago 2025), and generalization test set (NIBRS).
To make the data suitable for Logistic Regression, we perform the following transformations:
1. **Cyclical Encoding**: We convert `hour`, `month`, and `weekday` into sine and cosine values to preserve their periodic nature.
2. **One-Hot Encoding**: Categorical variables like `location_name` and `offense_category_name` are converted into binary matrices.
3. **Strict Feature Alignment**: We use the training set's feature columns as the baseline to align the test sets. Any missing categorical features in the test sets are padded with 0s. This ensures strict matrix dimensionality consistency during cross-dataset generalization.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/it5006/M2/Datasets'

path_chicago = f'{base_path}/Chicago'
path_nibrs = f'{base_path}/NIBRS/CA-2024'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import numpy as np

# Train: Chicago 2015-2024
df_train = pd.read_csv(f"{path_chicago}/Extracted_Crimes_2015-2024_aligned.csv")
# Test 1 (Local): Chicago 2025
df_test_local = pd.read_csv(f"{path_chicago}/Extracted_Crimes_2025_aligned.csv")
# Test 2 (Generalization): NIBRS 2024
df_test_gen = pd.read_csv(f"{path_nibrs}/NIBRS_aligned.csv")

print(f"Original Train Shape: {df_train.shape}")
print(f"Original Local Test Shape: {df_test_local.shape}")
print(f"Original Generalization Test Shape: {df_test_gen.shape}\n")

Original Train Shape: (2506726, 11)
Original Local Test Shape: (235060, 11)
Original Generalization Test Shape: (1161668, 11)



In [9]:
# Define feature engineering function
def preprocess_features(df):
    df_processed = df.copy()

    # Cyclical Encoding
    df_processed['hour_sin'] = np.sin(2 * np.pi * df_processed['hour'] / 24.0)
    df_processed['hour_cos'] = np.cos(2 * np.pi * df_processed['hour'] / 24.0)

    df_processed['month_sin'] = np.sin(2 * np.pi * df_processed['month'] / 12.0)
    df_processed['month_cos'] = np.cos(2 * np.pi * df_processed['month'] / 12.0)

    df_processed['weekday_sin'] = np.sin(2 * np.pi * df_processed['weekday'] / 7.0)
    df_processed['weekday_cos'] = np.cos(2 * np.pi * df_processed['weekday'] / 7.0)

    # Drop unused and original temporal columns
    # Note: 'year' and 'day' are dropped to prevent overfitting to specific past dates
    cols_to_drop = ['year', 'month', 'day', 'hour', 'weekday']
    df_processed.drop(columns=[c for c in cols_to_drop if c in df_processed.columns], inplace=True)

    return df_processed

In [10]:
# Extract X and y
target_col = 'is_arrested'

y_train = df_train[target_col]
X_train_raw = preprocess_features(df_train.drop(columns=[target_col]))

y_test_local = df_test_local[target_col]
X_test_local_raw = preprocess_features(df_test_local.drop(columns=[target_col]))

y_test_gen = df_test_gen[target_col]
X_test_gen_raw = preprocess_features(df_test_gen.drop(columns=[target_col]))

In [11]:
# One-Hot Encoding & Strict Alignment
categorical_cols = ['location_name', 'crime_against', 'offense_category_name']

X_train = pd.get_dummies(X_train_raw, columns=categorical_cols, drop_first=True)
X_test_local = pd.get_dummies(X_test_local_raw, columns=categorical_cols, drop_first=True)
X_test_gen = pd.get_dummies(X_test_gen_raw, columns=categorical_cols, drop_first=True)

X_test_local = X_test_local.reindex(columns=X_train.columns, fill_value=0)
X_test_gen = X_test_gen.reindex(columns=X_train.columns, fill_value=0)

# Ensure all features are float/int
X_train = X_train.astype(float)
X_test_local = X_test_local.astype(float)
X_test_gen = X_test_gen.astype(float)

print("=== Feature Alignment Successful ===")
print(f"Final X_train Shape: {X_train.shape}")
print(f"Final X_test_local Shape: {X_test_local.shape}")
print(f"Final X_test_gen Shape: {X_test_gen.shape}")

=== Feature Alignment Successful ===
Final X_train Shape: (2506726, 43)
Final X_test_local Shape: (235060, 43)
Final X_test_gen Shape: (1161668, 43)


In [13]:
# Single Source of Truth
X_train.to_csv(f'{path_chicago}/X_train_final.csv', index=False)
y_train.to_csv(f'{path_chicago}/y_train_final.csv', index=False)

X_test_local.to_csv(f'{path_chicago}/X_test_local_final.csv', index=False)
y_test_local.to_csv(f'{path_chicago}/y_test_local_final.csv', index=False)

X_test_gen.to_csv(f'{path_nibrs}/X_test_gen_final.csv', index=False)
y_test_gen.to_csv(f'{path_nibrs}/y_test_gen_final.csv', index=False)